# 10 · qa — 5지선다와 그 근거

장면에 대한 객관식 질문입니다. 원본이 답과 함께 풀이(rationale)를 제공하고, 그 풀이의 수치를 라벨로 재계산해 검증했습니다.

이 노트북은 네 가지를 확인합니다 — **어떤 원시 데이터에서**, **어떤 코드를 거쳐**, **무엇이 입력으로 들어가고**, **빌드된 파일이 그 코드와 일치하는지**. GPU 는 필요 없습니다.

In [ ]:
import os, sys, json, textwrap
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import numpy as np
import pandas as pd

from datatools import paths

ITEMS = os.path.join(paths.COMMON_DIR, "instruct_items_tasks01_06.parquet")
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 80)
wrap = lambda s, i="   ": textwrap.fill(str(s), 94, initial_indent=i,
                                        subsequent_indent=i)
VARIANTS = ['qa']
print("variants:", VARIANTS)

## 1. 어떤 원시 데이터에서 오는가

| 아카이브 | 주기 | 읽는 것 |
|---|---|---|
| `10_radar_vision_qa/qa_train/*.json` | 클립당 약 20문항 | question, options(A~E), answer, rationale, verification |
| `10_radar_vision_qa/qa_gt/*.json` | 139 클립 2,019문항 | 사람 검수본. 평가 전용이고 자동 수정을 적용하지 않았습니다 |
| `egomotion / obstacle.offline` | 10 Hz | rationale 의 수치를 재계산해 대조하는 데 쓰입니다 |

## 2. 어떤 코드를 거치는가

실행 순서입니다.

| 단계 | 하는 일 |
|---|---|
| `qa_claims.extract(rationale)` | 문장 단위로 수치 주장을 뽑음: ego_pos, ego_speed, agent_pos, agent_speed, distance, future_pos |
| `verify_qa` | 각 주장을 라벨에서 재계산해 비교 |
| `correct_qa_numbers` | safe 판정만 라벨값으로 교체 |
| `drop_bad_qa_items` | 고칠 수 없는 항목 제거 |
| `flag_qa_verification` | verification 필드 부착 |

In [ ]:
import inspect
for mod in ('qa_claims', 'verify_qa', 'correct_qa_numbers',
            'drop_bad_qa_items', 'flag_qa_verification'):
    m = __import__('datatools.' + mod, fromlist=[mod])
    print(f'{mod:22s} {(m.__doc__ or chr(10)).strip().splitlines()[0][:78]}')

## 3. 입력으로 무엇이 들어가는가

**비전 20장 · 레이더 20스캔 (20초, 1 Hz) · ego 질문 프레임까지**

입력 창은 태스크마다 다릅니다. 로더의 `WINDOWS` 표가 그것을 정하고, 레이더는 항상 20스캔이라 창의 길이가 바뀌면 샘플링 속도가 따라 바뀝니다 — 인코더 입력 모양은 변하지 않습니다.

In [ ]:
from training.instruct_data import WINDOWS, INSTANT_TASKS, WINDOW_TASKS
for v in VARIANTS:
    for name in (v, v + "_cot"):
        if name in WINDOWS:
            secs, hz, frames = WINDOWS[name]
            print(f"{name:26s} 창 {secs}초 · 레이더 {hz} Hz × 20스캔 · 비전 {frames}장")
        elif name in INSTANT_TASKS:
            print(f"{name:26s} 순간 — 비전 1장 · 레이더 20스캔/1초 · ego 1")
        else:
            print(f"{name:26s} 클립 전체 — 비전 20장 · 레이더 20스캔/20초")

## 4. 예제 10건 — 원시 입력째로

`notebooks/example_data/` 에 테스크마다 10건이 들어 있습니다. **parquet 도 원본 아카이브도 필요 없습니다.**

| 경로 | 내용 |
|---|---|
| `gen_data/<task>.jsonl` | LLM 학습에 쓰이는 아이템 그대로 — instruction, ego, 정답, 근거 |
| `raw/<task>/NN/frames/` | 그 아이템에 실제로 들어가는 프레임 |
| `raw/<task>/NN/radar.npz` | 그 창의 레이더 반사점 (패딩 제거) |

평문 변형과 CoT 변형은 같은 아이템입니다 — CoT 정답의 `answer` 필드가 평문 정답과 글자 그대로 같아서, 한 건이 둘을 모두 보여줍니다.

In [ ]:
import glob
EX = os.path.abspath(os.path.join(os.getcwd(), "..", "example_data"))
examples = {}
for v in VARIANTS:
    path = os.path.join(EX, "gen_data", v + ".jsonl")
    examples[v] = [json.loads(l) for l in open(path)] if os.path.exists(path) else []
    print(f"{v:26s} {len(examples[v]):>2}건")
rows = [{"id": r["id"], "clip": r["clip_id"][:8], "프레임": r["n_frames"],
         "레이더 점": r["radar_points"], "정답 길이": len(r["target"]),
         "근거 길이": len(r["rationale"])}
        for v in VARIANTS for r in examples[v]]
pd.DataFrame(rows)

한 건을 통째로 봅니다. instruction 이 출력 형식을 고르고, 근거가 그 형식의 답으로 이어집니다.

In [ ]:
pool = examples[VARIANTS[0]]
# 리그에 전방 레이더가 없는 클립이 17,130개 있고 예제에도 섞인다. 아래 그림이
# 보여줄 것이 있도록 반사점이 있는 건을 고르되, 없으면 그대로 쓰고 밝힌다.
r = next((x for x in pool if x["radar_points"]), pool[0])
if not r["radar_points"]:
    print("!! 이 예제는 전방 레이더가 없는 클립입니다 (반사점 0개)")
print("=" * 96)
print(f"{r['id']}   clip {r['clip_id'][:8]}   {r['sensors']}")
print(f"창: {r['window']}")
print()
print("instruction:"); print(wrap(r["instruction"].replace(chr(10), " | ")))
print("ego:");         print(wrap(r["ego"]))
print("answer (GT):"); print(wrap(r["target"]))
print("rationale:");   print(wrap(r["rationale"]))

그 아이템에 실제로 들어가는 프레임과 레이더 반사점입니다. 레이더는 자차 기준 좌표(x 전방, y 좌)이고 색이 시선속도 — 정지 물체는 자차 속도의 음수로 모여 보입니다.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

d = os.path.join(EX, r["raw"])
paths = sorted(glob.glob(os.path.join(d, "frames", "*.jpg")))
z = np.load(os.path.join(d, "radar.npz"))
pts, ch = z["points"].astype(np.float32), [str(c) for c in z["channels"]]
print(f"프레임 {len(paths)}장 · 반사점 {len(pts)}개 · 채널 {ch}")

fig, axes = plt.subplots(1, len(paths) + 1, figsize=(3.1 * (len(paths) + 1), 2.6))
axes = np.atleast_1d(axes)
for ax, p in zip(axes, paths):
    ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p), fontsize=8)
    ax.axis("off")
ax = axes[-1]
# 그림 안 글자는 ASCII 로 둔다. matplotlib 의 기본 폰트에 한글 글리프가 없어
# 라벨이 네모로 나오고, 폰트 등록을 요구하면 노트북이 기계를 가린다.
if len(pts):
    s = ax.scatter(pts[:, ch.index("y")], pts[:, ch.index("x")], s=2,
                   c=pts[:, ch.index("radial_velocity")], cmap="coolwarm")
    ax.set_title(f"radar: {r['radar_scans']} scans, {len(pts)} pts", fontsize=8)
    ax.invert_xaxis(); fig.colorbar(s, ax=ax, label="radial m/s")
else:
    ax.text(0.5, 0.5, "no returns", ha="center", va="center")
    ax.set_title("clip has no forward radar", fontsize=8)
ax.set_xlabel("y left (m)"); ax.set_ylabel("x forward (m)")
plt.tight_layout(); plt.show()

## 5. 빌드된 파일의 실제 아이템

여기서부터는 parquet 이 있는 기계에서만 돕니다. 위의 예제가 빌드 전체와 같은 규칙으로 만들어졌는지 확인하는 절입니다.

In [ ]:
from training.instruct_data import load_items
items = load_items(("qa",), "train")[:1] + load_items(("qa_cot",), "train")[:1]
for it in items:
    print("=" * 96)
    print(f"{it['task']}   clip {it['clip_id'][:8]}  frame {it['frame']}")
    print("Q:"); print(wrap(it["prompt"].replace(chr(10), " | ")))
    print("A:"); print(wrap(it["target"]))

## 6. CoT — 근거가 답을 만드는가

`_cot` 변형은 `{"rationale": ..., "answer": ...}` 입니다. **근거를 따라가면 답이 나와야** 합니다. 나오지 않으면 그 사슬은 잘못된 것이고, 보상을 걸면 모델이 그 잘못된 사슬을 배웁니다.

In [ ]:
from training.instruct_data import load_items
it = load_items(("qa_cot",), "train")[0]
d = json.loads(it["target"])
print("R:"); print(wrap(d["rationale"]))
print("A:"); print(wrap(d["answer"]))

## 7. 보상

평가 채점기에서 유도했습니다. 정답을 그대로 넣으면 1.0 이 나와야 하고, 내용을 망가뜨리면 떨어져야 합니다.

In [ ]:
from training.task_scorers import reward_for
from training.instruct_data import load_items
rows = []
for name in ("qa", "qa_cot"):
    fn = reward_for(name)
    t = load_items((name,), "train")[0]["target"]
    wrong = "A" if "A" not in t[:3] else "B"
    rows.append({"task": name, "reward": fn.__name__,
                 "정답": round(fn(t, t), 3),
                 "다른 글자": round(fn(wrong, t), 3)})
pd.DataFrame(rows)

## 8. 데이터 양

`val` 은 `train` 에 합쳐져 있습니다 — 클립 분할이 train 86,607 / val 54,163 / test 37,121 인데, 모델 선택은 `test` 에서 하므로 검증용 3분의 1이 쓰이지 않고 있었습니다.

In [ ]:
from collections import Counter
from training.instruct_data import load_items
names = [v for v in VARIANTS] + [v + "_cot" for v in VARIANTS]
rows = []
for split in ("train", "test"):
    c = Counter(i["task"] for i in load_items(tuple(names), split))
    for n in names:
        rows.append({"task": n, "split": split, "items": c.get(n, 0)})
pd.DataFrame(rows).pivot(index="task", columns="split", values="items")

## 9. 이 태스크에서 내린 결정과 근거

**두 세트를 합쳤다**

기존 1,999 클립과 새로 받은 8,000 클립은 클립이 하나도 겹치지 않아 중복 제거 없이 병합했습니다. 9,999 클립 195,874 문항입니다.

**자체 검증과 수정**

불일치 주장 중 서술 맥락(safe)만 라벨값으로 바꾸고, 계산에 쓰이거나 보기와 연동된 것은 항목째 제거했습니다. rationale 은 사실 목록이 아니라 논증이라, 한 숫자를 바꾸면 다음 줄이 틀려집니다.

**검증기가 틀렸던 지점**

사람 검수본의 거리 주장이 처음 58.6% 로 나왔는데, 추출기가 'X-position is 349.69m' 를 거리로 오인한 탓이었습니다. 고친 뒤 82.1% 입니다.

**CoT 는 모순이 확인된 것만 제외**

'agrees' 만 쓰면 62% 를 버리는데, 그 대부분은 틀린 것이 아니라 대조할 숫자가 없는 정성 추론입니다. 숫자가 아예 없는 것은 13.7% 뿐입니다.